# HW3: Regression and Classification

In this assignment you will preprocess the dataset and perform some basic regression and classification tasks. The learning outcome of this part is to know how one can pre-process a real-world dataset and perform a supervised learning task, and to understand some of the fundamental mechanisms behind these tasks.

##  Grading:

Pass/Fail.

To Pass this HW you need to provide a complete and correct solution, passing all the tests.

## OUTLINE:

Data pre-processing, regression task and classification task

1. Reading the files
2. Missing Values
3. Imputing categorical variables
4. Imputing numerical variables
5. Classification with Decision Tree, single split
6. Classification with Decision Tree, Cross validation
7. Interpretation of the results

## Important instructions:

Each function you make will be considered during the grading, so it is important to strictly follow input and output instructions stated in the skeleton code.

You must not change the names of the functions, since, if you do, the tests will fail.

Since this Homework is, in part, focused on having you implement creative solutions to impute missing data, if at any point of the homework you will use functions like fillna(), SimpleImputer(), IterativeImputer(), or packages like fancyimpute, missingpy, or similar, you will fail a test designed to spot these packages. Please, try to avoid circumventing this rule, since een if you manage to pass the homework, a similar task might be in the exam, and there you would be spotted for sure.

## Homework Scenario: Cleaning and Preparing Heart Disease Data

You have recently joined the **Data Science and Analytics Unit** at the *Global Health Institute (GHI)*, a non-profit organization focused on improving cardiovascular disease diagnosis through data-driven research.  

A junior data analyst from your team, **Franco**, sends you a message:

> “Hey, welcome to the team! We’re preparing a predictive model to help doctors identify patients at risk of heart disease using clinical data from several hospitals.  
>   
> We have two related datasets:
> - **Cleveland dataset** → this will be used for **training and validation**
> - **Hungary dataset** → this will serve as our **independent test set**
>
> Unfortunately, it looks like something went wrong during the data collection process: some values appear to have been **corrupted or lost**. Before we can train any classification model, we need to **inspect and clean the data**, handle **missing or inconsistent values**, and make sure it’s ready for modeling. I'm completely lost and I have a lot of other work, can you please help me with the cleaning and with creating some baselines classification models?”

Your task is to **analyze and clean the datasets** before **building a classifier** to predict whether a patient has heart disease.

In [104]:
# these are the libraries that you will need throughout the assignment
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from matplotlib.colors import ListedColormap

from HW import *

RSEED = 8

## *1.* Reading the files

### `Task: Read the datasets from the 'datasets' folder. Use the files called cleveland.csv and hungary.csv that you have downloaded in this archive.`

## Heart Disease Dataset — Column Descriptions

Someone has changed the names of some columns in the dataset, so make sure to use this description and refer to it for the "allowed" values.

Common sense is useful when evaluating some of the features: for example, in this dataset there is no column called weight, but, if there was one, since we are talking about humans and not ethereal beings, if a patient had a value of 0 in the weight column, this value could be due to a typo, or corrupted, and would need to be cleaned in some way.

| **Column** | **Description** |
|-------------|-----------------|
| **Age** | Age of the patient (in years). This dataset only includes adult patients. |
| **Sex** | Biological sex of the patient: `1 = male`, `0 = female`. |
| **ChestPainType** | Type of chest pain experienced: <br>• `1` = typical angina <br>• `2` = atypical angina <br>• `3` = non-anginal pain <br>• `4` = asymptomatic. |
| **RestBP** | Resting blood pressure (in mm Hg) measured on admission to the hospital. |
| **Chol** | Serum cholesterol level (in mg/dl). |
| **FBS** | Fasting blood sugar: `1` if fasting blood sugar > 120 mg/dl, otherwise `0`. |
| **RestECG** | Resting electrocardiographic results: <br>• `0` = normal <br>• `1` = ST-T wave abnormality <br>• `2` = showing probable or definite left ventricular hypertrophy. |
| **MaxHR** | Maximum heart rate achieved during the exercise test. |
| **ExAng** | Exercise-induced angina: `1` = yes, `0` = no. |
| **Oldpeak** | ST depression induced by exercise relative to rest (a measure of exercise-induced ischemia). |
| **Slope** | Slope of the peak exercise ST segment: <br>• `1` = upsloping <br>• `2` = flat <br>• `3` = downsloping. |
| **Ca** | Number of major vessels (0–3) colored by fluoroscopy (a measure of blood flow). |
| **Thal** | Thalassemia test result: <br>• `3` = normal <br>• `6` = fixed defect <br>• `7` = reversible defect. |
| **Num** | Diagnosis of heart disease (target variable): <br>`0` = no heart disease, `1–4` = presence of heart disease with increasing severity. |


In [105]:
from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

In [106]:
# From the folder 'datasets', read the files cleveland.csv and hungary.csv into the dataframes cleveland and test, respectively.

cleveland = pd.read_csv('/content/cleveland.csv')  # change this
test = pd.read_csv('/content/hungary.csv')       # change this

In [107]:
cleveland.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

In [108]:
test.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

In [109]:
# You can uncomment this to inspact the datasets
cleveland.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
1,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,3
2,222.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
3,58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,2
4,51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,2


In [110]:
(cleveland['Ca']=='?').sum()

np.int64(4)

In [111]:
(cleveland['Thal']=='?').sum()

np.int64(2)

In [112]:
test.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,47,0,2,140,257,0,0,135,0,1.0,1,?,?,0
1,52,1,4,112,342,0,1,96,1,1.0,2,?,?,1
2,41,0,2,125,184,0,-1,-1,0,0.0,?,?,?,0
3,58,1,4,135,222,0,0,100,0,0.0,?,?,?,0
4,54,0,2,140,309,?,1,140,0,0.0,?,?,?,0


In [113]:
# if you want to see information about the dataset, uncomment:
cleveland.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,55.052805,0.693069,3.442244,131.623762,3543.082508,0.148515,1.277228,246.953795,0.343234,1.039604,1.597360,0.937294
std,16.236574,0.599270,5.080393,17.549467,57434.537877,0.356198,5.210380,1715.122158,0.540737,1.161075,0.622104,1.228536
min,-1.000000,-1.000000,1.000000,94.000000,-234.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000
25%,47.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,132.500000,0.000000,0.000000,1.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,240.000000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,274.500000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,2.000000
max,222.000000,7.000000,90.000000,200.000000,1000000.000000,1.000000,90.000000,30000.000000,5.000000,6.200000,3.000000,4.000000


In [114]:
# if you want to see information about the dataset, uncomment:
test.describe()

,Age,Sex,ChestPainType,RestBP,Oldpeak,Num
count,293.000000,293.000000,293.000000,293.000000,293.000000,293.000000
mean,48.006826,0.733788,2.986348,132.583618,0.588055,0.361775
std,11.173903,0.486937,0.965049,17.607128,0.909554,0.481336
min,1.000000,0.000000,1.000000,92.000000,0.000000,0.000000
25%,42.000000,0.000000,2.000000,120.000000,0.000000,0.000000
50%,49.000000,1.000000,3.000000,130.000000,0.000000,0.000000
75%,54.000000,1.000000,4.000000,140.000000,1.000000,1.000000
max,170.000000,4.000000,4.000000,200.000000,5.000000,1.000000


In [115]:
cleveland.isna().sum()

,0
Age,0
Sex,0
ChestPainType,0
RestBP,0
Chol,0
FBS,0
RestECG,0
MaxHR,0
ExAng,0
Oldpeak,0


In [116]:
test.isna().sum()

,0
Age,0
Sex,0
ChestPainType,0
RestBP,0
Chol,0
FBS,0
RestECG,0
MaxHR,0
ExAng,0
Oldpeak,0


In [117]:
test.shape

(293, 14)

In [118]:
cleveland.dtypes

,0
Age,float64
Sex,float64
ChestPainType,float64
RestBP,float64
Chol,float64
FBS,float64
RestECG,float64
MaxHR,float64
ExAng,float64
Oldpeak,float64


In [119]:
test.dtypes

,0
Age,int64
Sex,int64
ChestPainType,int64
RestBP,int64
Chol,object
FBS,object
RestECG,object
MaxHR,object
ExAng,object
Oldpeak,float64


In [120]:
(test['Slope'] == '?').sum()

np.int64(188)

In [121]:
(test['Ca'] == '?').sum()

np.int64(289)

## *2.* Missing values

### `Task: use the function clean_data from the HW.py file to get a clean version of the cleveland and test dataframes.`

In [122]:
def clean_data(df):
    """
    Cleans the Cleveland heart disease dataset strictly based on the provided description:
    - Replaces '?' with NaN
    - Converts all columns to numeric dtype
    - Replaces invalid/out-of-spec values with NaN:
        * categorical columns outside allowed values
        * continuous columns with negative values
    - Returns cleaned DataFrame and missing value counts
    """

    df = df.copy()

    # Define columns
    categorical_columns = [
        'Sex', 'ChestPainType', 'FBS', 'RestECG',
        'ExAng', 'Slope', 'Ca', 'Thal'
    ]
    numerical_columns = [
        'Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak'
    ]

    # --- Step 1: Replace '?' with NaN ---
    df = df.replace('?', np.nan)

    # --- Step 2: Convert all columns to numeric (safe conversion) ---
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # --- Step 3a: Validate categorical columns ---
    valid_values = {
        'Sex': [0, 1],
        'ChestPainType': [1, 2, 3, 4],
        'FBS': [0, 1],
        'RestECG': [0, 1, 2],
        'ExAng': [0, 1],
        'Slope': [1, 2, 3],
        'Ca': [0, 1, 2, 3],
        'Thal': [3, 6, 7],
        'Num': [0, 1, 2, 3, 4]
    }

    for col in categorical_columns + ['Num']:
        if col in df.columns:
            df.loc[~df[col].isin(valid_values[col]), col] = np.nan

    # --- Step 3b: Validate continuous numeric columns ---
    for col in numerical_columns:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # --- Step 4: Missing value count ---
    missing_values_count = df.isna().sum().to_dict()

    return df, missing_values_count


In [123]:
# Write your code here
# cleveland_cleaned, missing_values_cleveland = pd.DataFrame(), {} # change this
# test_cleaned, missing_values_test = pd.DataFrame(), {} # change this

cleveland_cleaned, missing_values_cleveland = clean_data(cleveland) # change this
test_cleaned, missing_values_test = clean_data(test) # change this
print(missing_values_test)
print(missing_values_cleveland)

{'Age': 0, 'Sex': 1, 'ChestPainType': 0, 'RestBP': 0, 'Chol': 23, 'FBS': 8, 'RestECG': 2, 'MaxHR': 2, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 188, 'Ca': 289, 'Thal': 264, 'Num': 0}
{'Age': 1, 'Sex': 2, 'ChestPainType': 1, 'RestBP': 0, 'Chol': 1, 'FBS': 0, 'RestECG': 2, 'MaxHR': 1, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 1, 'Ca': 5, 'Thal': 3, 'Num': 0}


In [124]:
test_cleaned.isna().sum()

,0
Age,0
Sex,1
ChestPainType,0
RestBP,0
Chol,23
FBS,8
RestECG,2
MaxHR,2
ExAng,1
Oldpeak,0


In [125]:
cleveland_cleaned.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

## *3.* Imputing categorical variables

At the beginning of this file you can find the names of the columns and a description of their contents.

Determine which columns are categorical, and set their type to object.

Determine which columns are numerical, and set their type accordingly.

Do not include the target column in any of these lists!

In [126]:
categorical_columns = [
    'Sex',           # 0 = female, 1 = male
    'ChestPainType', # 1–4
    'FBS',           # 0/1
    'RestECG',       # 0–2
    'ExAng',         # 0/1
    'Slope',         # 1–3
    'Ca',            # 0–3
    'Thal'           # 3,6,7
]      # change this
numerical_columns = [
    'Age',     # continuous
    'RestBP',  # continuous
    'Chol',    # continuous
    'MaxHR',   # continuous
    'Oldpeak'  # continuous
]
target = 'Num'

In [127]:
cleveland_cleaned['Age'].min()

4.0

### ` Task: Split the cleveland_cleaned dataframe in a train and a validation set, using train_test_split from sklearn. `

The train set must be called train, the validation set must be called val. The size of the validation set must be 30% of the total size of the cleveland_cleaned dataframe. Use shuffle=True and stratify the split based on y_cleveland. Make sure that both train and val are dataframes, and that the columns have the correct names. Reset the indexes of all four the dataframes, using drop=True.

In [128]:
# Split the data into X and y, where X contains the features and y contains the target variable.
X_cleveland = cleveland_cleaned.drop(columns=['Num'])  # change this
y_cleveland = cleveland_cleaned['Num']  # change this

X_test = test_cleaned.drop(columns=['Num'])     # change this
y_test = test_cleaned['Num']      # change this

# For test data, just reset index
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [129]:
X_train, X_val, y_train, y_val = train_test_split(
    X_cleveland,
    y_cleveland,
    test_size=0.3,
    shuffle=True,
    stratify=y_cleveland,
    random_state=42
) # change this


# Reset indexes to keep consistent DataFrames
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)


In [130]:
print(type(X_train))

<class 'pandas.core.frame.DataFrame'>


In [131]:
# # if you want to see information about the split dataset, uncomment:
X_train.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
0,46.0,1.0,4.0,120.0,249.0,0.0,2.0,144.0,0.0,0.8,1.0,0.0,7.0
1,52.0,1.0,2.0,128.0,205.0,1.0,0.0,184.0,0.0,0.0,1.0,0.0,3.0
2,62.0,0.0,3.0,130.0,263.0,0.0,0.0,97.0,0.0,1.2,2.0,1.0,7.0
3,58.0,1.0,3.0,140.0,211.0,1.0,2.0,165.0,0.0,0.0,1.0,0.0,3.0
4,50.0,0.0,4.0,110.0,254.0,0.0,2.0,159.0,0.0,0.0,1.0,0.0,3.0


In [132]:
# # if you want to see information about the split dataset, uncomment:
X_val.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
0,56.0,0.0,2.0,140.0,NaN,0.0,2.0,153.0,0.0,1.3,2.0,0.0,3.0
1,47.0,1.0,3.0,130.0,253.0,0.0,0.0,179.0,0.0,0.0,1.0,0.0,3.0
2,61.0,1.0,4.0,148.0,203.0,0.0,0.0,161.0,0.0,0.0,1.0,1.0,7.0
3,59.0,1.0,3.0,126.0,218.0,1.0,0.0,134.0,0.0,2.2,2.0,1.0,6.0
4,43.0,1.0,4.0,150.0,247.0,0.0,0.0,171.0,0.0,1.5,1.0,0.0,3.0


In [133]:
X_train.min()

,0
Age,4.0
Sex,0.0
ChestPainType,1.0
RestBP,94.0
Chol,126.0
FBS,0.0
RestECG,0.0
MaxHR,0.0
ExAng,0.0
Oldpeak,0.0


In [134]:
# To make the classification task easier, transform the target variable into a binary variable.
# If the target variable is 0, it should remain 0riable is more than 0, . If the target vait should be transformed into 1.
# Using list comprehension
y_train = pd.DataFrame([0 if i == 0 else 1 for i in y_train], columns=['Num'])
y_val   = pd.DataFrame([0 if i == 0 else 1 for i in y_val], columns=['Num'])
y_test  = pd.DataFrame([0 if i == 0 else 1 for i in y_test], columns=['Num'])

In [135]:
X_train.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
count,212.000000,211.000000,212.000000,212.000000,212.000000,212.000000,210.000000,211.000000,211.000000,212.000000,212.000000,208.000000,209.000000
mean,54.566038,0.668246,3.198113,131.580189,4963.301887,0.183962,1.000000,289.881517,0.350711,1.040094,1.599057,0.600962,4.760766
std,13.909333,0.471963,0.927922,16.602637,68663.304324,0.388370,0.992797,2055.226274,0.478327,1.120700,0.611578,0.926997,1.946444
min,4.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,47.000000,0.000000,3.000000,120.000000,212.000000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,241.500000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,275.250000,0.000000,2.000000,167.000000,1.000000,1.650000,2.000000,1.000000,7.000000
max,197.000000,1.000000,4.000000,200.000000,1000000.000000,1.000000,2.000000,30000.000000,1.000000,6.200000,3.000000,3.000000,7.000000


### ` Task: use the impute_missing_categorical function from the HW.py file to impute the missing data from the categorical features in your dataframes. `

In [136]:
# Task 3: Categorical Features Imputation

def impute_missing_categorical(df_train, df_val, df_test, categorical_columns):
    """
    Task: Categorical Features Imputation
    --------------------------------------
    This function should handle missing values in categorical columns using appropriate techniques.
    We will skip scaling and encoding to keep things simple.

    Instructions:
    - Create subsets of the input DataFrames (train, validation, test) with only the categorical columns.
    - Use KNNImputer with k=5 and weights set to 'distance' to fill missing values in the categorical columns.
    - Ensure that the imputed values are approximated to the nearest value in the original dataset for each column, to avoid artifacts like decimal values.
    - If any "new value" is equidistant from two original values, choose the smaller one.
    - Add the column names to the resulting DataFrames after imputation.
    - The imputed dataframes should only contain the categorical columns.

    Parameters:
    df_train (pd.DataFrame): The training DataFrame.
    df_val (pd.DataFrame): The validation DataFrame.
    df_test (pd.DataFrame): The test DataFrame.
    categorical_columns (list): A list of column names corresponding to categorical features.

    Returns:
    pd.DataFrame: The training DataFrame with imputed categorical features.
    pd.DataFrame: The validation DataFrame with imputed categorical features.
    pd.DataFrame: The test DataFrame with imputed categorical features.
    """

    X_train_cat = df_train[categorical_columns].copy()
    X_val_cat = df_val[categorical_columns].copy()
    X_test_cat = df_test[categorical_columns].copy()

    imputer = KNNImputer(n_neighbors=5, weights='distance')
    imputer.fit(X_train_cat)

    # --- Step 3: Transform all sets ---
    X_train_imputed = imputer.transform(X_train_cat)
    X_val_imputed = imputer.transform(X_val_cat)
    X_test_imputed = imputer.transform(X_test_cat)

    for i, col in enumerate(categorical_columns):
        valid_values = np.sort(df_train[col].dropna().unique())  # original values
        # Function to snap each imputed value to nearest valid value
        def snap_to_nearest(value):
            diffs = np.abs(valid_values - value)
            min_diff = diffs.min()
            nearest_values = valid_values[diffs == min_diff]
            return nearest_values.min()  # pick smaller if tie
        X_train_imputed[:, i] = np.vectorize(snap_to_nearest)(X_train_imputed[:, i])
        X_val_imputed[:, i] = np.vectorize(snap_to_nearest)(X_val_imputed[:, i])
        X_test_imputed[:, i] = np.vectorize(snap_to_nearest)(X_test_imputed[:, i])

    # --- Step 5: Convert back to DataFrames with correct column names ---
    X_train_imputed = pd.DataFrame(X_train_imputed, columns=categorical_columns)
    X_val_imputed = pd.DataFrame(X_val_imputed, columns=categorical_columns)
    X_test_imputed = pd.DataFrame(X_test_imputed, columns=categorical_columns)

    return X_train_imputed, X_val_imputed, X_test_imputed



In [137]:
# Write your code here
X_train_imputed_cat, X_val_imputed_cat, X_test_imputed_cat = impute_missing_categorical(X_train, X_val, X_test, categorical_columns)

In [138]:
X_train_imputed_cat.isna().sum()

,0
Sex,0
ChestPainType,0
FBS,0
RestECG,0
ExAng,0
Slope,0
Ca,0
Thal,0


## *4.* Imputing numerical variables

` Task: use the impute_missing_numeric function from the HW.py file to impute the missing data from the numeric features in your dataframes. `

In [148]:
import numpy as np
from sklearn.linear_model import Lasso

def impute_numerical_features(df_train, df_val, df_test, numerical_columns):
    """
    Iterative Lasso-based imputer for numerical features.
    Fills missing values in train, val, and test using only models trained on non-missing train data for each numeric column.
    Always imputes the column with the fewest missing values next.
    Uses a fallback mean imputation on any remaining missing values.
    """

    train_num = df_train[numerical_columns].copy()
    val_num = df_val[numerical_columns].copy()
    test_num = df_test[numerical_columns].copy()

    # Save original indices for restoration at end
    train_idx = train_num.index
    val_idx = val_num.index
    test_idx = test_num.index

    train_imputed = train_num.copy()
    val_imputed = val_num.copy()
    test_imputed = test_num.copy()

    while (train_imputed.isnull().any().any() or
           val_imputed.isnull().any().any() or
           test_imputed.isnull().any().any()):
        missing_counts = train_imputed.isnull().sum()
        cols_with_missing = missing_counts[missing_counts > 0]
        if len(cols_with_missing) == 0:
            break  # No more missing columns left

        col_to_impute = cols_with_missing.sort_values().index[0]
        not_missing = train_imputed[col_to_impute].notnull()
        X_train_fit = train_imputed.loc[not_missing].drop(columns=[col_to_impute])
        y_train_fit = train_imputed.loc[not_missing, col_to_impute]

        if len(X_train_fit) > 0:
            model = Lasso(alpha=0.01, max_iter=2000)
            model.fit(X_train_fit.values, y_train_fit.values)

            missing_train = train_imputed[col_to_impute].isnull()
            if missing_train.any():
                X_missing_train = train_imputed.loc[missing_train].drop(columns=[col_to_impute])
                if len(X_missing_train) > 0:
                    train_imputed.loc[missing_train, col_to_impute] = model.predict(X_missing_train.values)

            missing_val = val_imputed[col_to_impute].isnull()
            if missing_val.any():
                X_missing_val = val_imputed.loc[missing_val].drop(columns=[col_to_impute])
                if len(X_missing_val) > 0:
                    val_imputed.loc[missing_val, col_to_impute] = model.predict(X_missing_val.values)

            missing_test = test_imputed[col_to_impute].isnull()
            if missing_test.any():
                X_missing_test = test_imputed.loc[missing_test].drop(columns=[col_to_impute])
                if len(X_missing_test) > 0:
                    test_imputed.loc[missing_test, col_to_impute] = model.predict(X_missing_test.values)

    # Fallback: fill remaining NaNs using train column mean
    for col in numerical_columns:
        col_mean = train_imputed[col].mean()
        train_imputed[col].fillna(col_mean, inplace=True)
        val_imputed[col].fillna(col_mean, inplace=True)
        test_imputed[col].fillna(col_mean, inplace=True)

    # Restore original order
    train_imputed = train_imputed.loc[train_idx]
    val_imputed = val_imputed.loc[val_idx]
    test_imputed = test_imputed.loc[test_idx]

    return train_imputed, val_imputed, test_imputed


In [149]:
# Impute numerical features using iterative Lasso
X_train_imputed_num, X_val_imputed_num, X_test_imputed_num = impute_numerical_features(
    df_train=X_train,
    df_val=X_val,
    df_test=X_test,
    numerical_columns=numerical_columns
)

/tmp/ipython-input-3091682997.py:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_imputed[col].fillna(col_mean, inplace=True)
/tmp/ipython-input-3091682997.py:64: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

In [152]:
X_val_imputed_num.isna().sum()

,0
Age,0
RestBP,0
Chol,0
MaxHR,0
Oldpeak,0


### ` Task: use the merge_imputed function from the HW.py file to merge your imputed dataframes. `

In [161]:
def merge_imputed(df_cat, df_num):
    """
    Task: Merge Imputed DataFrames
    -------------------------------
    This function should merge the imputed categorical and numerical DataFrames.

    Instructions:
    - Merge the imputed categorical and numerical DataFrames on their indexes.
    - Ensure that the resulting DataFrame contains all columns from both input DataFrames.

    Parameters:
    df_cat (pd.DataFrame): The DataFrame with imputed categorical features.
    df_num (pd.DataFrame): The DataFrame with imputed numerical features.

    Returns:
    pd.DataFrame: The merged DataFrame containing both categorical and numerical features.
    """
    merged = pd.concat([df_cat, df_num], axis=1)
    return merged

In [162]:
# Merge the train_imputed_cat and train_imputed_num datasets. Call the resulting dataset X_train_imputed.
# Merge the val_imputed_cat and val_imputed_num datasets. Call the resulting dataset X_val_imputed.
# Merge the test_imputed_cat and test_imputed_num datasets. Call the resulting dataset X_test_imputed.

# Write your code here
# Merge imputed numerical and categorical datasets by columns
X_train_imputed = merge_imputed(X_train_imputed_cat, X_train_imputed_num)
X_val_imputed = merge_imputed(X_val_imputed_cat, X_val_imputed_num)
X_test_imputed = merge_imputed(X_test_imputed_cat, X_test_imputed_num)


In [163]:
X_test_imputed.isna().sum()

,0
Sex,0
ChestPainType,0
FBS,0
RestECG,0
ExAng,0
Slope,0
Ca,0
Thal,0
Age,0
RestBP,0


## *5.* Classification, using a single split

### ` Use the function train_and_evaluate_single_split to produce classification results for your test set.`

In [167]:
# Task 5: Classification Using a Single Split
def train_and_evaluate_single_split(X_train, X_val, y_train, y_val, model, hp):
    """
    Task: Classification Using a Single Split
    ------------------------------------------
    This function should train a classification pipeline on the training set and evaluate it on the validation set, using the provided parameters.

    Instructions:
    - Create a classification pipeline. It should include:
        - A OneHotEncoder for categorical features (handle_unknown='ignore').
        - A StandardScaler for numerical features.
        - The provided classification model.
    - Use ColumnTransformer to apply the appropriate transformations to categorical and numerical features.
    - Set the model parameters using the provided parameters dictionary.
    - Train the model using the training data (X_train, y_train).
    - Evaluate the model on the validation data (X_val, y_val) using F1 score.
    - Return the evaluation results (F1 score) for the given parameters combination.

    Parameters:
    X_train (pd.DataFrame): The training feature set.
    X_val (pd.DataFrame): The validation feature set.
    y_train (pd.Series): The training labels.
    y_val (pd.Series): The validation labels.
    model: The classification model to train.
    hp (dict): A dictionary of hyperparameters to set for the model.

    Returns:
    dict: A dictionary containing two keys: 'params' (training parameters) and 'F1 scores' (F1 score). Each key should have the correct value.
    """
    model.set_params(**hp)

    # Define preprocessing for categorical and numerical columns
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', StandardScaler(), numerical_columns)
    ])

    # Full pipeline
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', model)
    ])

    # Train
    pipeline.fit(X_train, y_train.values.ravel())

    # Predict on validation set
    y_pred = pipeline.predict(X_val)

    # Evaluate F1 score (binary classification)
    f1 = f1_score(y_val, y_pred)

    return {'params': hp, 'F1 scores': f1}

In [168]:
# The hyperparameters for the tree should be:
# criterion: ['gini', 'entropy']
# max_depth: [3, 5, 10]
# The hyperparameters for the logistic regression should be:
# penalty: ['l1', 'l2']
# C: [0.1, 10]
# solver: ['liblinear']

# For each combination of hyperparameters, train a classification pipeline using your function.


from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

hyperparameters_tree = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10]
}# change this
hyperparameters_logreg = {
    'penalty': ['l1', 'l2'],
    'C': [0.1, 10],
    'solver': ['liblinear']
} # change this
performance_df = pd.DataFrame(columns=['params', 'F1 scores'])

# create a list from the grid of hyperparameters for each model, and create the models.

start = time.time() # DO NOT CHANGE/DELETE THIS LINE

for number in range(1, 11): # change this
    # call your function here, then concat the results to performance_df
    # --- Decision Tree ---
    for criterion in hyperparameters_tree['criterion']:
        for max_depth in hyperparameters_tree['max_depth']:
            hp = {'criterion': criterion, 'max_depth': max_depth}
            dt_model = DecisionTreeClassifier()
            result = train_and_evaluate_single_split(
                X_train_imputed, X_val_imputed, y_train, y_val,
                model=dt_model,
                hp=hp
            )
            performance_df = pd.concat([performance_df, pd.DataFrame([result])], ignore_index=True)

    # --- Logistic Regression ---
    for penalty in hyperparameters_logreg['penalty']:
        for C in hyperparameters_logreg['C']:
            for solver in hyperparameters_logreg['solver']:
                hp = {'penalty': penalty, 'C': C, 'solver': solver}
                lr_model = LogisticRegression()
                result = train_and_evaluate_single_split(
                    X_train_imputed, X_val_imputed, y_train, y_val,
                    model=lr_model,
                    hp=hp
                )
                performance_df = pd.concat([performance_df, pd.DataFrame([result])], ignore_index=True)

end = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with a single split: ', end - start) # DO NOT CHANGE/DELETE THIS LINE

/tmp/ipython-input-608974348.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  performance_df = pd.concat([performance_df, pd.DataFrame([result])], ignore_index=True)


Time elapsed to run the hyperparameter tuning with a single split:  1.6790130138397217


In [170]:
performance_df.sort_values(by='F1 scores', ascending=False)

,params,F1 scores
18,"{'penalty': 'l2', 'C': 0.1, 'solver': 'libline...",0.810127
28,"{'penalty': 'l2', 'C': 0.1, 'solver': 'libline...",0.810127
8,"{'penalty': 'l2', 'C': 0.1, 'solver': 'libline...",0.810127
38,"{'penalty': 'l2', 'C': 0.1, 'solver': 'libline...",0.810127
68,"{'penalty': 'l2', 'C': 0.1, 'solver': 'libline...",0.810127
...,...,...
73,"{'criterion': 'entropy', 'max_depth': 3}",0.657534
93,"{'criterion': 'entropy', 'max_depth': 3}",0.657534
80,"{'criterion': 'gini', 'max_depth': 3}",0.657534
45,"{'criterion': 'entropy', 'max_depth': 10}",0.651163


In [173]:
performance_df.iloc[18].params

{'penalty': 'l2', 'C': 0.1, 'solver': 'liblinear'}

In [ ]:
# 	{'criterion': 'entropy', 'max_depth': 3}	0.657534

In [ ]:
# {'penalty': 'l2', 'C': 0.1, 'solver': 'liblinear'}	0.810127

In [178]:
# Concatenate the train and validation datasets. Call the resulting datasets X and y.
# X = pd.DataFrame()  # change this
# y = pd.DataFrame()  # change this

X = pd.concat([X_train_imputed, X_val_imputed], ignore_index=True)
# Concatenate targets
y = pd.concat([y_train, y_val], ignore_index=True)
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Write your code here
best_hp = {'penalty': 'l2', 'C': 0.1, 'solver': 'liblinear'}  # replace with actual best
best_model = LogisticRegression(**best_hp)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', StandardScaler(), numerical_columns)
    ]
)
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', best_model)
])
y = y.squeeze()
pipeline.fit(X, y)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Sex', 'ChestPainType',
                                                   'FBS', 'RestECG', 'ExAng',
                                                   'Slope', 'Ca', 'Thal']),
                                                 ('num', StandardScaler(),
                                                  ['Age', 'RestBP', 'Chol',
                                                   'MaxHR', 'Oldpeak'])])),
                ('model', LogisticRegression(C=0.1, solver='liblinear'))])

## *6.* Classification with Decision Tree using Cross Validation

### ` Use the function train_and_evaluate_cross_validation to produce classification results for your test set.`

In [ ]:
# 1. Use the same hyperparameters from the previous task.
# 2. Create a dataframe to store the performance of the model with cross-validation, containing the columns 'params' and 'Average F1 scores'
# 3. You can reuse the parameter grids from the previous step.
# 4. Run your function for each combination of hyperparameters, using 5-fold cross-validation.
# 5. Concatenate the results to the dataframe created in step 2.



X = [[5,6], [10,11], [15,16], [20,21], [25,26], [30,31], [35,36], [40,41], [45,46], [50,51]]    # Delete this line
y = [0,1,0,1,0,1,0,1,0,1]                                                                       # Delete this line

X = pd.DataFrame(X)                                                                             # Delete this line
y = pd.DataFrame(y)                                                                             # Delete this line


# DO NOT FORGET TO DELETE THE PREVIOUS LINES. They are only to make the empty assignment run without errors,
# but they will destroy the data you need.

performance_df_cv = pd.DataFrame(columns=['params', 'F1 scores'])

start_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

# call your function here, then concat the results to performance_df_cv

end_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with Cross Validation: ', end_CV - start_CV) # DO NOT CHANGE/DELETE THIS LINE


In [ ]:
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

## *7.* Interpretation of the results

### ` Which model performs the best? `

Write your explanation here. Delete this text.

### ` Task: use the best model to produce predictions on the test set, then calculate the F1 score on the test set. What do you notice? `

### ` What is a possible explanation? `